# Module 2: Data Preprocessing with Snowpark ML

**Snowpark ML Fundamentals - Week 1 | Lunch & Learn**

---

## Learning Objectives
- Create and explore a synthetic dataset for ML
- Use `snowflake.ml.modeling.preprocessing` for distributed transformations
- Build feature engineering pipelines
- Understand that ALL preprocessing runs inside Snowflake

## Key Concept
> **Snowpark ML preprocessing mirrors scikit-learn's API** but executes distributed
> inside the Snowflake warehouse. No data leaves Snowflake.

---

In [11]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2.1 Setup & Create Sample Data

In [12]:
import sys
sys.path.insert(0, '../src')

from snowpark_fundamentals.session import create_session
from snowpark_fundamentals.data.sample_data import (
    create_customer_churn_dataset,
    get_dataset_summary,
)

session = create_session(env_path='../.env')

# Generate synthetic customer churn data (runs in Snowflake)
df = create_customer_churn_dataset(session, n_rows=5000)
print(f"Dataset shape: {df.count()} rows x {len(df.columns)} columns")
df.show(5)

Dataset shape: 5000 rows x 12 columns
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"AGE"  |"PLAN_TYPE"  |"TENURE_MONTHS"  |"MONTHLY_CHARGES"  |"SUPPORT_TICKETS"  |"USAGE_HOURS_PER_WEEK"  |"CONTRACT_TYPE"  |"PAYMENT_METHOD"  |"NUM_DEPENDENTS"  |"TOTAL_CHARGES"  |"CHURNED"  |
----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|0              |73     |STANDARD     |77.0             |256.0              |4                  |42                      |TWO_YEAR         |MAILED_CHECK      |0                 |19712.0          |0          |
|1              |21     |STANDARD     |105.0            |114.0              |4                  |39                      |MONT

In [13]:
# Churn distribution
summary = get_dataset_summary(df)
summary.show()

-------------------------------------------------------------------------------------------------------
|"CHURNED"  |"COUNT"  |"AVG_AGE"  |"AVG_TENURE"       |"AVG_MONTHLY_CHARGES"  |"AVG_SUPPORT_TICKETS"  |
-------------------------------------------------------------------------------------------------------
|0          |3890     |46.319023  |62.95089974293059  |157.20925449871464     |4.710026               |
|1          |1110     |46.267568  |56.81891891891892  |164.7135135135135      |5.990991               |
-------------------------------------------------------------------------------------------------------



## 2.2 Numeric Feature Scaling

Snowpark ML provides distributed implementations of:
- `StandardScaler` - zero mean, unit variance
- `MinMaxScaler` - scale to [0, 1]
- `MaxAbsScaler`, `RobustScaler`, etc.

These work **identically to scikit-learn** but process data inside Snowflake.

In [14]:
from snowpark_fundamentals.preprocessing.transformers import scale_numeric_features

numeric_cols = ['AGE', 'TENURE_MONTHS', 'MONTHLY_CHARGES', 'TOTAL_CHARGES',
                'SUPPORT_TICKETS', 'USAGE_HOURS_PER_WEEK', 'NUM_DEPENDENTS']

# StandardScaler
df_scaled, scaler = scale_numeric_features(df, numeric_cols, method='standard')

# Show original vs scaled
df_scaled.select(
    'AGE', 'AGE_SCALED',
    'MONTHLY_CHARGES', 'MONTHLY_CHARGES_SCALED'
).show(5)

------------------------------------------------------------------------
|"AGE"  |"AGE_SCALED"   |"MONTHLY_CHARGES"  |"MONTHLY_CHARGES_SCALED"  |
------------------------------------------------------------------------
|73     |1.5963012768   |256.0              |1.2004394668078553        |
|21     |-1.5134852689  |114.0              |-0.5546468168881262       |
|22     |-1.4536816815  |204.0              |0.5577318136234114        |
|74     |1.6561048642   |187.0              |0.34761585008234314       |
|29     |-1.0350565696  |31.0               |-1.5805071094709886       |
------------------------------------------------------------------------



In [15]:
# MinMaxScaler for comparison
df_minmax, _ = scale_numeric_features(df, ['AGE', 'MONTHLY_CHARGES'], method='minmax')
df_minmax.select(
    'AGE', 'AGE_SCALED',
    'MONTHLY_CHARGES', 'MONTHLY_CHARGES_SCALED'
).describe().show()

-----------------------------------------------------------------------------------------------------
|"SUMMARY"  |"AGE"              |"AGE_SCALED"        |"MONTHLY_CHARGES"  |"MONTHLY_CHARGES_SCALED"  |
-----------------------------------------------------------------------------------------------------
|min        |18.0               |8e-18               |20.0               |0.0                       |
|max        |75.0               |1.0                 |300.0              |1.0                       |
|stddev     |16.72307731848418  |0.2933873211161666  |80.91579510911097  |0.28898498253253924       |
|count      |5000.0             |5000.0              |5000.0             |5000.0                    |
|mean       |46.3076            |0.4966245614035088  |158.8752           |0.4959828571428571        |
-----------------------------------------------------------------------------------------------------



## 2.3 Categorical Feature Encoding

Available encoders:
- `OneHotEncoder` - creates binary columns per category
- `OrdinalEncoder` - maps categories to integers
- `LabelEncoder` - single column encoding

In [16]:
from snowpark_fundamentals.preprocessing.transformers import encode_categorical_features

categorical_cols = ['PLAN_TYPE', 'CONTRACT_TYPE', 'PAYMENT_METHOD']

# Ordinal encoding (good for tree-based models)
df_encoded, encoder = encode_categorical_features(
    df, categorical_cols, method='ordinal'
)

df_encoded.select(
    'PLAN_TYPE', 'PLAN_TYPE_ENCODED',
    'CONTRACT_TYPE', 'CONTRACT_TYPE_ENCODED',
    'PAYMENT_METHOD', 'PAYMENT_METHOD_ENCODED',
).show(10)

-------------------------------------------------------------------------------------------------------------------------------
|"PLAN_TYPE"  |"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD"  |"PAYMENT_METHOD_ENCODED"  |
-------------------------------------------------------------------------------------------------------------------------------
|STANDARD     |2.0                  |TWO_YEAR         |2.0                      |MAILED_CHECK      |3.0                       |
|STANDARD     |2.0                  |MONTH_TO_MONTH   |0.0                      |MAILED_CHECK      |3.0                       |
|BASIC        |0.0                  |TWO_YEAR         |2.0                      |CREDIT_CARD       |1.0                       |
|BASIC        |0.0                  |ONE_YEAR         |1.0                      |BANK_TRANSFER     |0.0                       |
|PREMIUM      |1.0                  |MONTH_TO_MONTH   |0.0                      |BANK_TRANSFER     |0.0 

## 2.4 Feature Engineering

Creating domain-specific features that improve model performance:

In [17]:
from snowpark_fundamentals.preprocessing.feature_engineering import (
    create_derived_features,
    create_interaction_features,
)

# Create derived features
df_features = create_derived_features(df)

# Show new features
df_features.select(
    'CUSTOMER_ID', 'CHARGES_PER_MONTH_TENURE', 'TICKETS_PER_TENURE',
    'TENURE_BUCKET', 'HIGH_VALUE_FLAG', 'AT_RISK_FLAG'
).show(10)

----------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"CHARGES_PER_MONTH_TENURE"  |"TICKETS_PER_TENURE"  |"TENURE_BUCKET"  |"HIGH_VALUE_FLAG"  |"AT_RISK_FLAG"  |
----------------------------------------------------------------------------------------------------------------------------
|0              |256.0                       |0.05194805194805195   |LOYAL            |1                  |0               |
|1              |114.0                       |0.0380952380952381    |LOYAL            |0                  |0               |
|2              |204.0                       |0.05                  |LOYAL            |1                  |0               |
|3              |187.0                       |0.02564102564102564   |ESTABLISHED      |1                  |0               |
|4              |31.0                        |0.16                  |MEDIUM           |0                  |0               |


In [18]:
# Interaction features
df_interactions = create_interaction_features(
    df,
    col_pairs=[
        ('SUPPORT_TICKETS', 'MONTHLY_CHARGES'),
        ('AGE', 'TENURE_MONTHS'),
    ]
)

df_interactions.select(
    'SUPPORT_TICKETS', 'MONTHLY_CHARGES',
    'SUPPORT_TICKETS_X_MONTHLY_CHARGES',
    'AGE', 'TENURE_MONTHS', 'AGE_X_TENURE_MONTHS'
).show(5)

---------------------------------------------------------------------------------------------------------------------------------
|"SUPPORT_TICKETS"  |"MONTHLY_CHARGES"  |"SUPPORT_TICKETS_X_MONTHLY_CHARGES"  |"AGE"  |"TENURE_MONTHS"  |"AGE_X_TENURE_MONTHS"  |
---------------------------------------------------------------------------------------------------------------------------------
|4                  |256.0              |1024.0                               |73     |77.0             |5621.0                 |
|4                  |114.0              |456.0                                |21     |105.0            |2205.0                 |
|4                  |204.0              |816.0                                |22     |80.0             |1760.0                 |
|1                  |187.0              |187.0                                |74     |39.0             |2886.0                 |
|4                  |31.0               |124.0                                |29     |25.

## 2.5 Complete Preprocessing Pipeline

Combine scaling + encoding in a reusable pipeline:

In [19]:
from snowpark_fundamentals.preprocessing.transformers import build_preprocessing_pipeline

df_processed, transformers = build_preprocessing_pipeline(
    df=df,
    numeric_cols=numeric_cols,
    categorical_cols=categorical_cols,
    numeric_method='standard',
    categorical_method='ordinal',
)

print(f"Processed columns: {len(df_processed.columns)}")
print(f"Fitted transformers: {list(transformers.keys())}")
df_processed.show(5)

Processed columns: 22
Fitted transformers: ['scaler', 'encoder']
-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"PLAN_TYPE_ENCODED"  |"CONTRACT_TYPE_ENCODED"  |"PAYMENT_METHOD_ENCODED"  |"CUSTOMER_ID"  |"AGE"  |"PLAN_TYPE"  |"TENURE_MONTHS"  |"MONTHLY_CHARGES"  |"SUPPORT_TICKETS"  |"USAGE_HOURS_PER_WEEK"  |"CONTRACT_TYPE"  |"PAYMENT_METHOD"  |"NUM_DEPENDENTS"  |"TOTAL_CHARGES"  |"CHURNED"  |"AGE_SCALED"   |"TENURE_MONTHS_SCALED"  |"MONTHLY_CHARGES_SCALED"  |"TOTAL_CHARGES_SCALED"  |"SUPPORT_TICKETS_SCALED"  |"USAGE_HOURS_PER_WEEK_SCALED"  |"NUM_DEPENDENTS_SCALED"  |
-----------

## Key Takeaways

1. `snowflake.ml.modeling.preprocessing` mirrors **scikit-learn's API** (fit/transform)
2. Key difference: operates on **Snowpark DataFrames** using `input_cols`/`output_cols`
3. **All computation runs in Snowflake** - no data transfer to client
4. Supports StandardScaler, MinMaxScaler, OneHotEncoder, OrdinalEncoder, and more
5. Feature engineering uses **Snowpark DataFrame operations** (F.when, F.col, etc.)

---

**Next: [03 - Model Training](03_model_training.ipynb)**

In [10]:
session.close()